# Risk-Tiered Robo-Advisor — ML / Optimization Pipeline

**MGMT 59900 Big Data Analytics in the Cloud — Group 1 Robo-Advisor Project**
Author: Amit Jain, Emrah Surucu, Todd Thiel

End-to-end pipeline: CFP questionnaire scorecard → risk tier → ML-forecast volatility → shrinkage-stabilized covariance → bounded minimum-variance optimization → cost-aware backtest vs SPY. Results persisted to S3 / Athena for the QuickSight dashboard.

**Pipeline stages**
1. Load 5-asset panel (4 sleeve ETFs + SPY) from Athena
2. Feature engineering + forward-volatility target (lookahead-audited)
3. XGBoost volatility model vs naive baseline; per-asset evaluation → hybrid rule
4. Walk-forward volatility predictions at monthly rebalance dates
5. Ledoit-Wolf-shrunk correlations → covariance Σ = D·C·D
6. Bounded minimum-variance optimization (dim_risk_tier constraints)
7. Cost-aware backtest vs SPY
8. CFP scorecard → persona tier assignment
9. Persist results to S3 / Athena

## 0. Setup

Install dependencies (safe to re-run) and import. `cvxpy` is installed here too so the whole notebook runs top-to-bottom after a fresh kernel.

In [ ]:
%pip install -q awswrangler scikit-learn xgboost cvxpy

import warnings
warnings.filterwarnings("ignore")

import awswrangler as wr
import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.covariance import LedoitWolf
from xgboost import XGBRegressor
import cvxpy as cp

DATABASE = "robo_advisor_tt"
BUCKET = "mgmt59900-group1-robo-advisor-tt"

## 1. Load panel, engineer features, establish baseline

Pull the four sleeve ETFs from `fact_asset_class` and SPY from `fact_benchmark` (SPY serves as both the US-equity asset and the backtest benchmark). Build backward-looking features (trailing volatility, momentum) and a forward 30-day realized-volatility target. The feature/target boundary is causal — verified in the next cell.

Two views are kept: `train` (full per-asset history, for model training) and `train_common` (the window where all 5 assets coexist, ~2011+, for the optimizer's covariance).

In [ ]:
# ================= 1) LOAD: 4 sleeves + SPY =================
df = wr.athena.read_sql_query(
    'SELECT ticker, price_date, daily_return FROM fact_asset_class ORDER BY ticker, price_date',
    database=DATABASE, ctas_approach=False,
)
df["price_date"] = pd.to_datetime(df["price_date"])
df = df.dropna(subset=["daily_return"]).reset_index(drop=True)

spy = wr.athena.read_sql_query(
    "SELECT ticker, price_date, daily_return FROM fact_benchmark "
    "WHERE ticker = 'SPY' ORDER BY price_date",
    database=DATABASE, ctas_approach=False,
)
spy["price_date"] = pd.to_datetime(spy["price_date"])
spy = spy.dropna(subset=["daily_return"]).reset_index(drop=True)

panel = pd.concat([df, spy], ignore_index=True)
panel = panel.sort_values(["ticker", "price_date"]).reset_index(drop=True)
assert panel.duplicated(subset=["ticker", "price_date"]).sum() == 0
assert panel["daily_return"].notna().all()
print("panel:", panel.shape, "| assets:", sorted(panel["ticker"].unique()))

# ================= 2) FEATURES + TARGET =================
TRADING_DAYS = 252
VOL_WINDOWS = [21, 63]
MOM_WINDOWS = [21, 63, 126]
TARGET_FWD = 30

def build_features(g):
    g = g.sort_values("price_date").reset_index(drop=True)
    r = g["daily_return"]
    for w in VOL_WINDOWS:
        g[f"vol_{w}"] = r.rolling(w).std() * np.sqrt(TRADING_DAYS)
    for w in MOM_WINDOWS:
        g[f"mom_{w}"] = (1 + r).rolling(w).apply(np.prod, raw=True) - 1
    fwd = r.shift(-1).rolling(TARGET_FWD).std() * np.sqrt(TRADING_DAYS)
    g["target_fwd_vol_30"] = fwd.shift(-(TARGET_FWD - 1))
    return g

feat = panel.groupby("ticker", group_keys=False).apply(build_features)

# ================= 3) ASSEMBLE TRAIN TABLE =================
model_cols = ["vol_21", "vol_63", "mom_21", "mom_63", "mom_126", "target_fwd_vol_30"]
train = feat.dropna(subset=model_cols).reset_index(drop=True)

per_start = train.groupby("ticker")["price_date"].min()
per_end   = train.groupby("ticker")["price_date"].max()
common_start, common_end = per_start.max(), per_end.min()
train_common = train[(train["price_date"] >= common_start) &
                     (train["price_date"] <= common_end)].reset_index(drop=True)
print(f"train_full: {len(train)} | train_common: {len(train_common)} "
      f"({common_start.date()} -> {common_end.date()})")

# ================= 4) TIME SPLIT + BASELINE =================
FEATURES = ["vol_21", "vol_63", "mom_21", "mom_63", "mom_126"]
TARGET = "target_fwd_vol_30"

cutoff = train["price_date"].quantile(0.80)
tr = train[train["price_date"] <= cutoff].reset_index(drop=True)
te = train[train["price_date"] >  cutoff].reset_index(drop=True)
print(f"\nsplit at {cutoff.date()} | train {len(tr)} | test {len(te)}")

base_rmse = np.sqrt(mean_squared_error(te[TARGET], te["vol_21"]))
base_mae  = mean_absolute_error(te[TARGET], te["vol_21"])
print(f"BASELINE (vol_21 -> fwd vol):  RMSE={base_rmse:.6f}  MAE={base_mae:.6f}")

### 1a. Lookahead boundary audit

Hand-verify one row: the trailing feature window must end at *t* (inclusive) and the target window must start strictly after *t*. This is the single most important check in a volatility pipeline — a shift-by-one error here silently leaks future information.

In [ ]:
# Pick a ticker + a row with full history on both sides
tk = "SPY"
s = feat[feat["ticker"] == tk].sort_values("price_date").reset_index(drop=True)
i = 3000
row = s.loc[i]
t_date = row["price_date"]
print(f"Auditing {tk} at t = {t_date.date()} (row {i})\n")

manual_vol21 = s["daily_return"].iloc[i-20:i+1].std() * np.sqrt(252)
print(f"vol_21  pipeline={row['vol_21']:.6f}  manual={manual_vol21:.6f}  "
      f"match={np.isclose(row['vol_21'], manual_vol21)}")

manual_target = s["daily_return"].iloc[i+1:i+31].std() * np.sqrt(252)
print(f"target  pipeline={row['target_fwd_vol_30']:.6f}  manual={manual_target:.6f}  "
      f"match={np.isclose(row['target_fwd_vol_30'], manual_target)}")

feat_window_end   = s["price_date"].iloc[i]
target_window_start = s["price_date"].iloc[i+1]
print(f"\nfeature window ends:   {feat_window_end.date()}")
print(f"target window starts:  {target_window_start.date()}")
print(f"no overlap (target strictly after feature): {target_window_start > feat_window_end}")

## 2. XGBoost volatility model vs baseline

Fit XGBoost on the time-based split and compare against the naive persistence baseline (forward vol ≈ trailing 21-day vol). Volatility is persistent, so this is a strong baseline; the model must beat it to justify inclusion.

In [ ]:
X_tr, y_tr = tr[FEATURES], tr[TARGET]
X_te, y_te = te[FEATURES], te[TARGET]

model = XGBRegressor(
    n_estimators=300, max_depth=4, learning_rate=0.03,
    subsample=0.8, colsample_bytree=0.8, random_state=42, n_jobs=-1,
)
model.fit(X_tr, y_tr)

pred = model.predict(X_te)
model_rmse = np.sqrt(mean_squared_error(y_te, pred))
model_mae  = mean_absolute_error(y_te, pred)

print(f"BASELINE:  RMSE={base_rmse:.6f}  MAE={base_mae:.6f}")
print(f"XGBOOST:   RMSE={model_rmse:.6f}  MAE={model_mae:.6f}")
print(f"\nRMSE improvement: {(base_rmse-model_rmse)/base_rmse*100:+.1f}%")
print(f"MAE  improvement: {(base_mae-model_mae)/base_mae*100:+.1f}%")

print("\nfeature importance:")
for f, imp in sorted(zip(FEATURES, model.feature_importances_), key=lambda x: -x[1]):
    print(f"  {f:10s} {imp:.3f}")

### 2a. Per-asset evaluation → hybrid rule

The pooled improvement masks asset-class variation. Breaking error down per asset (vs each asset's own baseline, plus a level-normalized error) reveals the model beats baseline on high-volatility assets (SPY, GLD, VXUS) but hurts on near-deterministic ones (AGG, BIL). This motivates the **hybrid volatility rule**: ML where it wins, baseline where it doesn't.

In [ ]:
print(f"{'asset':6s} {'n':>5s} {'mean_vol':>9s} {'base_rmse':>10s} {'xgb_rmse':>9s} {'improve':>8s} {'norm_err':>9s}")
print("-" * 62)

for tk in sorted(te["ticker"].unique()):
    m = te["ticker"] == tk
    y = te.loc[m, TARGET]
    xgb_p = model.predict(te.loc[m, FEATURES])
    base_p = te.loc[m, "vol_21"]
    b_rmse = np.sqrt(mean_squared_error(y, base_p))
    x_rmse = np.sqrt(mean_squared_error(y, xgb_p))
    improve = (b_rmse - x_rmse) / b_rmse * 100 if b_rmse > 0 else 0
    norm = x_rmse / y.mean() if y.mean() > 0 else np.nan
    print(f"{tk:6s} {m.sum():5d} {y.mean():9.4f} {b_rmse:10.5f} {x_rmse:9.5f} {improve:+7.1f}% {norm:9.3f}")

print("-" * 62)
print(f"{'POOLED':6s} {len(te):5d} {te[TARGET].mean():9.4f} {base_rmse:10.5f} {model_rmse:9.5f} "
      f"{(base_rmse-model_rmse)/base_rmse*100:+7.1f}% {model_rmse/te[TARGET].mean():9.3f}")

## 3. Walk-forward volatility predictions

At each month-end rebalance date, refit the model on **strictly prior** data (no lookahead), then predict forward volatility. Apply the hybrid rule: ML-predicted vol for SPY/GLD/VXUS, baseline trailing vol for AGG/BIL. Produces the diagonal inputs for the covariance matrix.

In [ ]:
ML_ASSETS = ["SPY", "GLD", "VXUS"]      # model beats baseline
BASE_ASSETS = ["AGG", "BIL"]           # baseline wins -> use vol_21
MIN_TRAIN = 2000                        # min rows before trusting a refit

common_dates = feat[(feat["price_date"] >= common_start) &
                    (feat["price_date"] <= common_end)]["price_date"].drop_duplicates().sort_values()
cal = pd.DataFrame({"price_date": common_dates})
cal["ym"] = cal["price_date"].dt.to_period("M")
rebal_dates = cal.groupby("ym")["price_date"].max().tolist()   # last trading day each month
print(f"{len(rebal_dates)} candidate rebalance dates: "
      f"{rebal_dates[0].date()} -> {rebal_dates[-1].date()}")

records = []
skipped = 0
for t in rebal_dates:
    past = feat[feat["price_date"] < t].dropna(subset=FEATURES + [TARGET])
    if len(past) < MIN_TRAIN:
        skipped += 1
        continue
    m = XGBRegressor(n_estimators=300, max_depth=4, learning_rate=0.03,
                     subsample=0.8, colsample_bytree=0.8, random_state=42, n_jobs=-1)
    m.fit(past[FEATURES], past[TARGET])

    rows_t = feat[feat["price_date"] == t].dropna(subset=FEATURES)
    if not {"SPY","GLD","VXUS","AGG","BIL"}.issubset(set(rows_t["ticker"])):
        skipped += 1
        continue

    for _, r in rows_t.iterrows():
        tk = r["ticker"]
        if tk in ML_ASSETS:
            vol_hat = float(m.predict(r[FEATURES].values.reshape(1, -1))[0]); src = "ml"
        else:
            vol_hat = float(r["vol_21"]); src = "baseline"
        records.append({"rebal_date": t, "ticker": tk, "vol_hat": vol_hat, "source": src})

pred_vol = pd.DataFrame(records)
print(f"skipped {skipped} early/incomplete dates")
print(f"usable rebalance dates: {pred_vol['rebal_date'].nunique()}")
print(f"prediction rows: {len(pred_vol)}  (should be 5 x dates)")
counts = pred_vol.groupby("rebal_date")["ticker"].nunique()
print(f"all dates have 5 assets: {(counts == 5).all()}")
print("\nsample (first rebalance date):")
print(pred_vol[pred_vol["rebal_date"] == pred_vol["rebal_date"].min()].to_string(index=False))

## 4. Covariance construction: Σ = D · C · D

**C** — trailing 126-day correlation matrix, stabilized with Ledoit-Wolf shrinkage applied to *correlations only* (standardize returns to unit variance, then LedoitWolf yields a shrunk correlation matrix with analytic λ). **D** — diagonal of hybrid volatility predictions. Assemble Σ and verify positive semi-definiteness on every date (the injected hybrid diagonal is not assumed safe).

In [ ]:
CORR_WINDOW = 126

wide = panel.pivot(index="price_date", columns="ticker", values="daily_return").sort_index()
ASSETS = ["SPY", "AGG", "VXUS", "GLD", "BIL"]   # fixed order for all matrices
wide = wide[ASSETS]
print("wide returns shape:", wide.shape)

corr_mats = {}
skipped_c = 0
for t in pred_vol["rebal_date"].unique():
    win = wide.loc[:t].tail(CORR_WINDOW).dropna()
    if len(win) < CORR_WINDOW:
        skipped_c += 1
        continue
    z = (win - win.mean()) / win.std(ddof=0)     # unit variance -> shrunk CORRELATION
    lw = LedoitWolf().fit(z.values)
    corr_mats[t] = (pd.DataFrame(lw.covariance_, index=ASSETS, columns=ASSETS), lw.shrinkage_)

print(f"built {len(corr_mats)} correlation matrices; skipped {skipped_c}")

t_last = max(corr_mats.keys())
C_last, lam = corr_mats[t_last]
print(f"\n{t_last.date()}  shrinkage lambda = {lam:.3f}")
print("diagonal ~ 1:", np.allclose(np.diag(C_last), 1.0, atol=0.05))
print("PSD (min eigenvalue >= 0):", np.linalg.eigvalsh(C_last.values).min() >= -1e-8)
print("\nshrunk correlation matrix:")
print(C_last.round(3).to_string())

In [ ]:
sigma_mats = {}
psd_fail = []
for t in corr_mats.keys():
    C, _ = corr_mats[t]
    vrow = pred_vol[pred_vol["rebal_date"] == t].set_index("ticker")["vol_hat"]
    D = np.diag(vrow[ASSETS].values)              # hybrid vols, annualized
    Sigma = pd.DataFrame(D @ C.values @ D, index=ASSETS, columns=ASSETS)
    sigma_mats[t] = Sigma
    if np.linalg.eigvalsh(Sigma.values).min() < -1e-8:
        psd_fail.append(t)

print(f"assembled {len(sigma_mats)} covariance matrices")
print(f"PSD failures: {len(psd_fail)}")

t_last = max(sigma_mats.keys())
S = sigma_mats[t_last]
vrow = pred_vol[pred_vol["rebal_date"] == t_last].set_index("ticker")["vol_hat"][ASSETS]
print(f"\n{t_last.date()} - Sigma diagonal = vol^2:",
      np.allclose(np.diag(S.values), vrow.values**2))
print(f"min eigenvalue: {np.linalg.eigvalsh(S.values).min():.6e}")
print("\ncovariance matrix Sigma (annualized):")
print(S.round(4).to_string())

## 5. Risk-tier bounds and feasibility

Load `dim_risk_tier` (5 tiers × 5 asset classes, min/max weight bounds) and pre-validate feasibility: each tier's min-weights must sum to ≤ 1 and max-weights to ≥ 1, or the constraint set is infeasible.

In [ ]:
dt_cols = wr.catalog.table(database=DATABASE, table="dim_risk_tier")
print("columns:", list(dt_cols["Column Name"]))
print()
risk_tier = wr.athena.read_sql_query(
    "SELECT * FROM dim_risk_tier ORDER BY 1, 2",
    database=DATABASE, ctas_approach=False,
)
print(risk_tier.to_string(index=False))

In [ ]:
print("tier feasibility (min_sum <= 1 <= max_sum):")
for tk in sorted(risk_tier["tier_key"].unique()):
    t = risk_tier[risk_tier["tier_key"] == tk]
    lo, hi = t["min_weight"].sum(), t["max_weight"].sum()
    name = t["tier_name"].iloc[0]
    print(f"  tier {tk} {name:22s}: min_sum={lo:.2f}  max_sum={hi:.2f}  feasible={lo <= 1.0 <= hi}")

## 6. Bounded minimum-variance optimization

For each rebalance date × tier, solve `min wᵀΣw` subject to fully-invested, long-only, and per-asset tier bounds. The tier bounds are load-bearing: without them, minimum-variance collapses every tier into the same bond/cash portfolio. Verify all solves are optimal, tiers differentiate, and no bounds are violated.

In [ ]:
# Pre-build per-tier bound vectors in ASSETS order (explicit lookup, no positional assumption)
tier_bounds = {}
for tk in sorted(risk_tier["tier_key"].unique()):
    t = risk_tier[risk_tier["tier_key"] == tk].set_index("ticker")
    lo = np.array([t.loc[a, "min_weight"] for a in ASSETS])
    hi = np.array([t.loc[a, "max_weight"] for a in ASSETS])
    tier_bounds[tk] = (t["tier_name"].iloc[0], lo, hi)

def solve_min_var(Sigma, lo, hi):
    w = cp.Variable(len(ASSETS))
    prob = cp.Problem(cp.Minimize(cp.quad_form(w, cp.psd_wrap(Sigma))),
                      [cp.sum(w) == 1, w >= lo, w <= hi])
    prob.solve(solver=cp.OSQP, verbose=False)
    return prob.status, (w.value if w.value is not None else None)

records = []
nonopt = []
for t, Sigma in sigma_mats.items():
    S = Sigma.values
    for tk, (name, lo, hi) in tier_bounds.items():
        status, wv = solve_min_var(S, lo, hi)
        if status != "optimal" or wv is None:
            nonopt.append((t, tk, status)); continue
        wv = np.clip(wv, 0, None); wv = wv / wv.sum()
        rec = {"rebal_date": t, "tier_key": tk, "tier_name": name}
        rec.update({a: wv[i] for i, a in enumerate(ASSETS)})
        records.append(rec)

weights = pd.DataFrame(records)
print(f"solved {len(weights)} portfolios (expect {len(sigma_mats)*5} = dates x 5 tiers)")
print(f"non-optimal solves: {len(nonopt)}")

t_last = weights["rebal_date"].max()
print(f"\nweights at {t_last.date()} (each row sums to 1):")
show = weights[weights["rebal_date"] == t_last][["tier_name"] + ASSETS].copy()
show[ASSETS] = show[ASSETS].round(3)
print(show.to_string(index=False))

viol = 0
for tk, (name, lo, hi) in tier_bounds.items():
    sub = weights[weights["tier_key"] == tk]
    for i, a in enumerate(ASSETS):
        if (sub[a] < lo[i] - 1e-4).any() or (sub[a] > hi[i] + 1e-4).any():
            viol += 1
print(f"\nbound violations across all portfolios: {viol}")

## 7. Cost-aware backtest vs SPY

Weights are held between rebalances (realistic monthly drift) and reset at each month-end, with a 10 bps turnover cost applied to the difference between the **drifted** portfolio and the new target. Reports gross and net metrics per tier against SPY buy-and-hold.

In [ ]:
COST_RATE = 0.0010   # 10 bps per unit turnover, one-way

bt_start = weights["rebal_date"].min()
rets = wide[wide.index >= bt_start].copy().dropna()
rebal_list = sorted(weights["rebal_date"].unique())
tiers = sorted(weights["tier_key"].unique())

def run_backtest(tier_key, apply_costs=True):
    wtbl = weights[weights["tier_key"] == tier_key].set_index("rebal_date")[ASSETS]
    dates = rets.index
    reb_arr = np.array(rebal_list, dtype="datetime64[ns]")
    reb_idx = np.searchsorted(reb_arr, dates.values.astype("datetime64[ns]"), side="right") - 1
    port_ret = np.zeros(len(dates))
    w_cur = None; last_reb = -1
    for k, d in enumerate(dates):
        ri = reb_idx[k]
        if ri < 0:
            continue
        r = rets.iloc[k].values
        if ri != last_reb:
            w_tgt = wtbl.loc[rebal_list[ri]].values
            if w_cur is not None and apply_costs:
                port_ret[k] -= np.abs(w_tgt - w_cur).sum() * COST_RATE   # target vs DRIFTED
            w_cur = w_tgt.copy(); last_reb = ri
        day = float(np.dot(w_cur, r))
        port_ret[k] += day
        w_cur = (w_cur * (1 + r)) / (1 + day)                            # drift
    return pd.Series(port_ret, index=dates)

def metrics(sr):
    eq = (1 + sr).cumprod()
    ann_ret = eq.iloc[-1] ** (252 / len(sr)) - 1
    ann_vol = sr.std() * np.sqrt(252)
    sharpe = ann_ret / ann_vol if ann_vol > 0 else np.nan
    dd = (eq / eq.cummax() - 1).min()
    return eq.iloc[-1] - 1, ann_ret, ann_vol, sharpe, dd

spy_sr = rets["SPY"]
rows = []
for tk in tiers:
    name = weights[weights["tier_key"] == tk]["tier_name"].iloc[0]
    net = run_backtest(tk, apply_costs=True); gro = run_backtest(tk, apply_costs=False)
    cn = metrics(net); cg = metrics(gro)
    rows.append({"tier": name, "cum_net": cn[0], "cum_gross": cg[0],
                 "ann_ret": cn[1], "ann_vol": cn[2], "sharpe": cn[3], "max_dd": cn[4],
                 "cost_drag": cg[0]-cn[0]})
bm = metrics(spy_sr)
rows.append({"tier": "SPY (benchmark)", "cum_net": bm[0], "cum_gross": bm[0],
             "ann_ret": bm[1], "ann_vol": bm[2], "sharpe": bm[3], "max_dd": bm[4], "cost_drag": 0})

results = pd.DataFrame(rows)
pd.set_option("display.float_format", lambda x: f"{x:.4f}")
print(f"Backtest {bt_start.date()} -> {rets.index.max().date()}  ({len(rets)} days, {len(rebal_list)} rebalances)\n")
print(results.to_string(index=False))

## 8. CFP questionnaire scorecard → persona tier assignment

Deterministic business-rules engine (not ML) mapping CFP answers to a risk tier. Three scoring drivers (horizon dominant 0-3, risk-reaction 0-2, capacity 0-2) sum to a base tier; three downward-only gates (horizon < 3yr → Conservative; no emergency fund → Mod-Conservative; liquidity event 1-5yr → Moderate) cap the result. Most-restrictive gate wins — `final_tier = min(scored, *caps)` — so a constraint can only lower risk, never raise it.

Five personas span all five final tiers and each gate fires at least once (P3 liquidity, P4 horizon, P5 emergency-fund).

In [ ]:
TIER_NAMES = {1: "conservative", 2: "moderate_conservative", 3: "moderate",
              4: "moderate_aggressive", 5: "aggressive"}

def score_horizon(years):          # Q4 - dominant driver, 0-3 pts
    if years < 3:   return 0
    if years <= 7:  return 1
    if years <= 15: return 2
    return 3

def score_reaction(r):             # Q6 - behavioral tolerance, 0-2 pts
    return {"sell": 0, "hold": 1, "buy_more": 2}[r]

def score_capacity(c):             # Q8 - financial capacity, 0-2 pts
    return {"unstable": 0, "stable": 1, "very_secure": 2}[c]

def score_to_tier(pts):            # 0-7 -> tier 1-5
    if pts <= 1: return 1
    if pts <= 3: return 2
    if pts == 4: return 3
    if pts <= 6: return 4
    return 5

def assign_tier(p):
    pts = score_horizon(p["horizon_years"]) + score_reaction(p["risk_reaction"]) \
        + score_capacity(p["income_stability"])
    scored = score_to_tier(pts)
    caps = [scored]; gate_notes = []
    if p["horizon_years"] < 3:
        caps.append(1); gate_notes.append("horizon<3yr->cap Conservative")
    if not p["has_emergency_fund"]:
        caps.append(2); gate_notes.append("no e-fund->cap Mod-Conservative")
    if p["liquidity_event_1_5yr"]:
        caps.append(3); gate_notes.append("liquidity event->cap Moderate")
    final = min(caps)                          # constraint overrides score
    return pts, scored, final, "; ".join(gate_notes) if gate_notes else "none"

personas = [
    {"persona_id": "P1", "label": "Young accumulator", "age": 32,
     "horizon_years": 30, "risk_reaction": "buy_more", "income_stability": "very_secure",
     "has_emergency_fund": True,  "liquidity_event_1_5yr": False, "primary_goal": "retirement"},
    {"persona_id": "P2", "label": "Mid-career balanced", "age": 45,
     "horizon_years": 15, "risk_reaction": "buy_more", "income_stability": "stable",
     "has_emergency_fund": True,  "liquidity_event_1_5yr": False, "primary_goal": "wealth building"},
    {"persona_id": "P3", "label": "Long horizon, tuition looming", "age": 40,
     "horizon_years": 20, "risk_reaction": "buy_more", "income_stability": "very_secure",
     "has_emergency_fund": True,  "liquidity_event_1_5yr": True,  "primary_goal": "retirement + college 2029"},
    {"persona_id": "P4", "label": "Pre-retiree, near-term need", "age": 60,
     "horizon_years": 2,  "risk_reaction": "sell", "income_stability": "stable",
     "has_emergency_fund": True,  "liquidity_event_1_5yr": False, "primary_goal": "capital preservation"},
    {"persona_id": "P5", "label": "High scorer, no safety net", "age": 28,
     "horizon_years": 25, "risk_reaction": "buy_more", "income_stability": "stable",
     "has_emergency_fund": False, "liquidity_event_1_5yr": False, "primary_goal": "wealth building"},
]

rows = []
for p in personas:
    pts, scored, final, notes = assign_tier(p)
    rows.append({**p, "score_pts": pts,
                 "scored_tier": scored, "scored_tier_name": TIER_NAMES[scored],
                 "final_tier": final, "final_tier_name": TIER_NAMES[final],
                 "gates_applied": notes})

dim_user_profile = pd.DataFrame(rows)
cols = ["persona_id", "label", "age", "horizon_years", "risk_reaction",
        "income_stability", "has_emergency_fund", "liquidity_event_1_5yr",
        "score_pts", "scored_tier_name", "final_tier_name", "gates_applied"]
print(dim_user_profile[cols].to_string(index=False))

## 9. Persist results to S3 / Athena

Write three tables as Parquet and register them in the `robo_advisor_tt` Athena database (under the `results/` prefix). These power the QuickSight dashboard:
- **dim_user_profile** — personas: CFP answers + score + assigned tier
- **fact_tier_weights** — long-form weight history (180 dates × 5 tiers × 5 assets)
- **fact_backtest** — daily equity curves + drawdown per tier and SPY benchmark

`mode="overwrite"` makes this idempotent — re-running replaces only these three result tables and touches nothing else.

In [ ]:
RESULTS_PREFIX = f"s3://{BUCKET}/results"

# ---- 1) Rebuild backtest capturing DAILY equity curves ----
def backtest_series(tier_key, apply_costs=True):
    sr = run_backtest(tier_key, apply_costs=apply_costs)
    eq = (1 + sr).cumprod()
    dd = eq / eq.cummax() - 1
    return pd.DataFrame({"date": sr.index, "daily_return": sr.values,
                         "equity": eq.values, "drawdown": dd.values})

bt_frames = []
for tk in tiers:
    name = weights[weights["tier_key"] == tk]["tier_name"].iloc[0]
    d = backtest_series(tk, apply_costs=True)
    d["tier_key"] = tk; d["tier_name"] = name
    bt_frames.append(d)
spy_eq = (1 + spy_sr).cumprod()
bt_frames.append(pd.DataFrame({"date": spy_sr.index, "daily_return": spy_sr.values,
                               "equity": spy_eq.values,
                               "drawdown": (spy_eq/spy_eq.cummax()-1).values,
                               "tier_key": 0, "tier_name": "SPY_benchmark"}))
fact_backtest = pd.concat(bt_frames, ignore_index=True)
print("fact_backtest:", fact_backtest.shape)

# ---- 2) Reshape weights to long form ----
fact_tier_weights = weights.melt(
    id_vars=["rebal_date", "tier_key", "tier_name"],
    value_vars=ASSETS, var_name="ticker", value_name="weight")
print("fact_tier_weights:", fact_tier_weights.shape)

# ---- 3) Persist all three ----
def persist(dfx, table):
    wr.s3.to_parquet(df=dfx, path=f"{RESULTS_PREFIX}/{table}/",
                     dataset=True, mode="overwrite", database=DATABASE, table=table)
    print(f"  wrote {table}: {len(dfx)} rows -> {RESULTS_PREFIX}/{table}/")

print("persisting...")
persist(dim_user_profile, "dim_user_profile")
persist(fact_tier_weights, "fact_tier_weights")
persist(fact_backtest, "fact_backtest")
print("done - all 3 tables registered in Athena database:", DATABASE)

### 9a. Verify tables are queryable

Confirm all three tables read back from Athena and the persona → tier → weights join (the dashboard's core relationship) returns the expected 900 weight rows per persona.


In [ ]:
for tbl in ["dim_user_profile", "fact_tier_weights", "fact_backtest"]:
    n = wr.athena.read_sql_query(f"SELECT COUNT(*) AS n FROM {tbl}",
                                 database=DATABASE, ctas_approach=False)["n"].iloc[0]
    print(f"{tbl}: {n} rows queryable")

print("\npersona -> assigned portfolio (final tier):")
q = """
SELECT p.persona_id, p.label, p.final_tier_name,
       COUNT(w.ticker) AS n_weight_rows
FROM dim_user_profile p
LEFT JOIN fact_tier_weights w
  ON p.final_tier_name = w.tier_name
GROUP BY p.persona_id, p.label, p.final_tier_name
ORDER BY p.persona_id
"""
print(wr.athena.read_sql_query(q, database=DATABASE, ctas_approach=False).to_string(index=False))

## 10. Create Summary Metrics Fact Table

In [ ]:
# Persist summary metrics table for the dashboard
fact_metrics = results.rename(columns={
    "tier": "tier_name", "cum_net": "cumulative_return",
    "ann_ret": "annualized_return", "ann_vol": "annualized_vol",
    "sharpe": "sharpe_ratio", "max_dd": "max_drawdown", "cost_drag": "cost_drag"
}).copy()

order_map = {"conservative":1,"moderate_conservative":2,"moderate":3,
             "moderate_aggressive":4,"aggressive":5,"SPY (benchmark)":6}
fact_metrics["sort_key"] = fact_metrics["tier_name"].map(order_map).fillna(9).astype(int)

wr.s3.to_parquet(df=fact_metrics, path=f"{RESULTS_PREFIX}/fact_metrics/",
                 dataset=True, mode="overwrite", database=DATABASE, table="fact_metrics")
print("wrote fact_metrics:", fact_metrics.shape)
print(fact_metrics.to_string(index=False))